# 14 Camera Sync Ingest QC Template

Template for camera synchronization ingest checks and QC summaries.


In [ ]:
import logging
import warnings

logging.getLogger("datajoint").setLevel(logging.WARNING)
warnings.filterwarnings("ignore", message="pkg_resources is deprecated as an API.*", category=UserWarning)

from adamacs.notebook_runtime import bootstrap_ingest_notebook

ctx = bootstrap_ingest_notebook(verbose=False)
repo_root = ctx.repo_root

import datajoint as dj


In [ ]:

scan_keys = (
    scan.Scan * session.Session * session.SessionUser * subject.User
    & f'initials = "{INITIALS}"'
    & f'session_datetime >= "{DATE_FROM}"'
).fetch("KEY")

print("scan keys:", len(scan_keys))


In [ ]:

records = []
for key in scan_keys:
    try:
        sync_event_count = len(event.Event & key & 'event_type LIKE "%sync%"')
    except Exception:
        sync_event_count = None

    records.append(
        {
            **key,
            "behavior_recording": len(event.BehaviorRecording & key),
            "video_recording": len(model.VideoRecordingNew & key),
            "camsync_recording": len(behavior.CamSyncRecording & key),
            "camera_timestamps": len(event.CameraTimestamps & key) if hasattr(event, "CameraTimestamps") else None,
            "sync_event_count": sync_event_count,
        }
    )

qc = pd.DataFrame(records)
qc


In [ ]:

needs_sync_attention = qc[
    (qc["behavior_recording"] > 0)
    & (
        (qc["camsync_recording"] == 0)
        | (qc["sync_event_count"].fillna(0) == 0)
    )
]

needs_sync_attention
